### Notebook to create BLOCK-T401 for Hexapod LUT in elevation with LaserTracker

This notebook sweeps three elevation points [10, 45, 85] and measures with the laser tracker the location of M2 and the camera at each of those elevations. V1 offers an alternative with more elevation points.

Created on: 2025-03-26

Author: Guillem Megias

In [ ]:
from lsst.ts.observing import ObservingBlock, ObservingScript 
from lsst.ts.aos.analysis import build_configuration_schema
import os

In [ ]:
current_path = os.getcwd()
block_number = 'T401'
name = "BLOCK-T401"
program = "BLOCK-T401v2"
reason = "LUT_elevation_hexapod_lasertracker"
constraints = []

### Define configuration schema

In [ ]:
# Define the configurable properties that we will use in the configuration schema
properties = {
    "azimuth": {
        "description": "Azimuth to use for the elevation sweep.",
        "type": "number",
        "default": 0
    }
}

# Build the configuration schema for BLOCK-404
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

### Define scripts and block

In [ ]:
m2_measure_script = ObservingScript(
    name="maintel/laser_tracker/measure.py",
    standard=True,
    parameters= dict(
        target="M2",
        program="$program",
        reason=reason,
    )
)

camera_measure_script = ObservingScript(
    name="maintel/laser_tracker/measure.py",
    standard=True,
    parameters= dict(
        target="Camera",
        program="$program",
        reason=reason
    )
)

stop_tracking_script = ObservingScript(
    name="maintel/stop_tracking.py",
    standard=True,
    parameters = dict()
)

In [ ]:
elevations = [10, 45, 85]
scripts = []

for elevation in elevations:
    track_target_script = ObservingScript(
        name="maintel/track_target.py",
        standard=True,
        parameters = dict(
            track_azel = dict(
                az = '$azimuth',
                el = elevation
            )
        )
    )

    scripts.append(track_target_script)
    scripts.append(camera_measure_script)
    scripts.append(m2_measure_script)
scripts.append(stop_tracking_script)

In [ ]:
block = ObservingBlock(
    name = name,
    program = program,
    configuration_schema=configuration_schema,
    scripts = scripts,
)

### Save configurable block

In [ ]:
block.model_dump_json(indent=2)

output_file_path = f'{current_path}/aos/ts_config_ocs/Scheduler/observing_blocks_maintel/AOS/LUTs/{program}.json'

with open(output_file_path, 'w') as file:
    file.write(block.model_dump_json(indent=2))